## Ingest one arXiv paper
Fetch the HTML version of a paper and split it into sections.

### Download the paper's HTML
Get the raw page from arXiv. `200` = success; most of the ~490k characters are markup, not text.

In [2]:
import httpx
arxiv_id = "2410.22360"
r = httpx.get(f"https://arxiv.org/html/{arxiv_id}", follow_redirects=True)
print(r.status_code, len(r.text))
print(r.text[:500])

200 489612
<!DOCTYPE html><html lang="en">
<head>
<meta http-equiv="content-type" content="text/html; charset=UTF-8">
<title>arxivDIGESTables: Synthesizing Scientific Literature into Tables using Language Models</title>
<!--Generated by LaTeXML oxide (version 0.7.6) http://dlmf.nist.gov/LaTeXML/.-->
<meta name="viewport" content="width=device-width, initial-scale=1, shrink-to-fit=no">
<link rel="stylesheet" href="/static/browse/0.3.4/css/arxiv-html-papers-20260823.css" type="text/css">
<script src="/static


## List the sections
Parse the HTML with BeautifulSoup and find every `ltx_section`. Abstract, references and appendices use other tags.

In [3]:
from bs4 import BeautifulSoup
soup = BeautifulSoup(r.text, "html.parser")
for sec in soup.find_all("section", class_="ltx_section"):
    print(sec.get("id"), "|", sec.find(class_="ltx_title").get_text(" ", strip=True))


S1 | 1 Introduction
S2 | 2 Creating arxivDIGESTables
S3 | 3 Literature Review Table Generation
S4 | 4 Experiments
S5 | 5 Developing an Automatic Metric
S6 | 6 Results
S7 | 7 Related Work
S8 | 8 Conclusion
Sx1 | Limitations
Sx2 | Ethical Considerations and Broader Impact


## Extract the text of each section
One record per section (id, title, text), plus the abstract. This list is what later steps will chunk and embed.

In [4]:
sections = []
abstract = soup.find(class_="ltx_abstract")
if abstract:
    sections.append({"id": "abstract", "title": "Abstract", "text": abstract.get_text(" ", strip=True)})
for sec in soup.find_all("section", class_="ltx_section"):
    title = sec.find(class_="ltx_title").get_text(" ", strip=True)
    sections.append({"id": sec.get("id"), "title": title, "text": sec.get_text(" ", strip=True)})
for s in sections:
    print(f"{s['title'][:40]:40} {len(s['text']):>7,} chars")

Abstract                                   1,428 chars
1 Introduction                             5,757 chars
2 Creating arxivDIGESTables                6,078 chars
3 Literature Review Table Generation       2,275 chars
4 Experiments                              2,216 chars
5 Developing an Automatic Metric           5,662 chars
6 Results                                  7,690 chars
7 Related Work                             2,957 chars
8 Conclusion                                 885 chars
Limitations                                1,086 chars
Ethical Considerations and Broader Impac   1,210 chars


## Inspect the text quality
Read one section to spot leftover junk: citation markers, math, captions, spacing.

In [6]:
print(sections[1]["text"][:1500])


1 Introduction Figure 1: Schematic of our literature review table generation task: (1) synthesize multiple input papers into a table with both (2) a schema (columns) and (3) values. Each row corresponds to an input paper. Figure 2: Side-by-side comparison of a reference literature review table from an ArXiv paper ( Liu et al., 2023 ) and a model-generated table given the same input papers.
The generated table has reconstructed two gold aspects:
the pink and blue aspects are the same, despite surface form differences (e.g., “Task” vs “Intended Application”).
The generated table has also proposed two novel aspects that are still relevant and useful, like “evaluation metric” ( green ) or “Annotation method” ( yellow ) not to be confused with reference table’s “Annotations”. Conducting literature reviews by reading and synthesizing information across a large set of documents is vital for scientists to stay abreast of their fields yet is increasingly laborious as the number of scientific pu

## Clean text
Remove figure captions and repeated titles, squeeze whitespace. Tables are kept on purpose (they hold results).


In [10]:
import copy

def clean_text(el):
    el = copy.copy(el)
    for fig in el.find_all("figure", class_="ltx_figure"):
        fig.decompose()
    for m in el.find_all("math"):
        m.replace_with(f" ${m.get('alttext', '')}$ ")
    title = el.find(class_="ltx_title")
    if title:
        title.decompose()
    return " ".join(el.get_text(" ").split())

sections = []
if abstract:
    sections.append({"id": "abstract", "title": "Abstract", "text": clean_text(abstract)})
for sec in soup.find_all("section", class_="ltx_section"):
    title = sec.find(class_="ltx_title").get_text(" ", strip=True)
    sections.append({"id": sec.get("id"), "title": title, "text": clean_text(sec)})

print(sections[1]["text"][:800])

Conducting literature reviews by reading and synthesizing information across a large set of documents is vital for scientists to stay abreast of their fields yet is increasingly laborious as the number of scientific publications grows exponentially Jinha (2010) ; Bornmann et al. (2021) . At the core of this sensemaking process is identifying a schema , a set of important aspects that are useful for comparing and contrasting prior literature Russell et al. (1993) . The results of this process are often presented in the form of literature review tables , whose rows are a set of papers and whose columns are a set of aspects that the papers share (Figure 1 ). In this work, we conceptualize the task of literature review table generation by decomposing it into two sub-tasks: (1) Schema-generatio


## Save the paper as JSON
Store id, title and clean sections in `data/papers/`, so later steps can load it without re-downloading.


In [8]:
import json
from pathlib import Path

out = Path("../data/papers") / f"{arxiv_id}.json"
out.parent.mkdir(parents=True, exist_ok=True)
paper = {"arxiv_id": arxiv_id, "title": soup.title.get_text(strip=True), "sections": sections}
out.write_text(json.dumps(paper, indent=2, ensure_ascii=False))
print(out.resolve(), out.stat().st_size, "bytes")

/Users/ibrahim/paper-companion/data/papers/2410.22360.json 36826 bytes


## Reuse on a second paper
Wrap the pipeline in `fetch_paper()` and test it on "Attention Is All You Need" (math + subsections).


In [12]:
def fetch_paper(arxiv_id):
    r = httpx.get(f"https://arxiv.org/html/{arxiv_id}", follow_redirects=True)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    sections = []
    abstract = soup.find(class_="ltx_abstract")
    if abstract:
        sections.append({"id": "abstract", "title": "Abstract", "text": clean_text(abstract)})
    for sec in soup.find_all("section", class_="ltx_section"):
        title = sec.find(class_="ltx_title").get_text(" ", strip=True)
        sections.append({"id": sec.get("id"), "title": title, "text": clean_text(sec)})
    return {"arxiv_id": arxiv_id, "title": soup.title.get_text(strip=True), "sections": sections}

paper2 = fetch_paper("1706.03762")
for s in paper2["sections"]:
    print(f"{s['title'][:40]:40} {len(s['text']):>7,} chars")
print(paper2["sections"][3]["text"][:800])

Abstract                                   1,139 chars
1 Introduction                             1,943 chars
2 Background                               1,840 chars
3 Model Architecture                       9,966 chars
4 Why Self-Attention                       3,845 chars
5 Training                                 2,228 chars
6 Results                                  7,248 chars
7 Conclusion                               1,216 chars
Attention Visualizations                       0 chars
Most competitive neural sequence transduction models have an encoder-decoder structure [ 5 , 2 , 35 ] . Here, the encoder maps an input sequence of symbol representations $(x_{1},...,x_{n})$ to a sequence of continuous representations $\mathbf{z}=(z_{1},...,z_{n})$ . Given $\mathbf{z}$ , the decoder then generates an output sequence $(y_{1},...,y_{m})$ of symbols one element at a time. At each step the model is auto-regressive [ 10 ] , consuming the previously generated symbols as additional input wh